# Screen Box videos for bioluminescent signals

This notebook lists videos in a public Box folder, downloads a small batch **one video at a time**, measures brightness with `scripts/brightnessOverTime.py`, and deletes each local video immediately after it is analyzed. Measurement CSVs are retained in `data/processed/brightness_parts/`, so completed videos can be skipped on later runs.

The video filename is interpreted as a UTC start time. Plots and the `time_panama` column use `America/Panama` (UTC−5). Start with one of the smaller files shown in the listing before processing a large batch.

In [53]:
# Main inputs
BOX_FOLDER_URL = "https://ucsb.box.com/s/g5dt84y2krfraipbbjjaligjh8u49qag"
BATCH_START = 0       # zero-based row in the sorted video table
BATCH_SIZE = 31        # keep this small; videos are downloaded one at a time
FRAME_STEP = 3        # 1 analyzes every frame; 2 or 3 gives a faster preview
MAX_WIDTH = 640       # downscale for speed; 0 measures at full resolution
TIMEZONE = "America/Panama"

# Spot-sensitive metric settings
TOP_FRACTION = 0.001
PERCENTILE = 99.9
BRIGHT_THRESHOLD = 200
VIDEO_REGION = "right"  # use "left", "right", or "full"
FORCE_REPROCESS = False   # set True only to redo existing right-camera results

# Plot ranges (values above these are visually clipped, not discarded)
MEAN_Y_AXIS_MAX = 0.05
BRIGHT_Y_AXIS_MAX = 50

In [54]:
import importlib.util
import json
import re
import sys
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "brightnessOverTime.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "brightnessOverTime.py"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "brightness_parts"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

spec = importlib.util.spec_from_file_location("brightness_over_time", SCRIPT_PATH)
brightness = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = brightness
spec.loader.exec_module(brightness)
print(f"Using {SCRIPT_PATH}")
print(f"Keeping measurement files in {RESULTS_DIR}")

Using /Users/oakley/Documents/GitHub/manta-analysis/scripts/brightnessOverTime.py
Keeping measurement files in /Users/oakley/Documents/GitHub/manta-analysis/data/processed/brightness_parts


In [ ]:
POST_DATA_RE = re.compile(r"Box\.postStreamData\s*=\s*(\{.*?\});", re.DOTALL)
VIDEO_EXTENSIONS = {"avi", "m4v", "mkv", "mov", "mp4", "mpg", "mpeg"}

def _read_url(url, attempts=4):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml",
    }
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read().decode("utf-8")
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as error:
            if attempt == attempts:
                status = getattr(error, "code", "network error")
                raise RuntimeError(f"Box request failed after {attempts} attempts (status: {status}): {url}") from error
            wait = 2 ** (attempt - 1)
            print(f"Box request failed ({error}); retrying in {wait} seconds...")
            time.sleep(wait)

def list_box_videos(shared_url):
    rows = []
    page = 1
    while True:
        separator = "&" if "?" in shared_url else "?"
        html = _read_url(f"{shared_url}{separator}page={page}")
        match = POST_DATA_RE.search(html)
        if not match:
            raise RuntimeError("Could not find Box folder metadata. Check that the link is public.")
        post_data = json.loads(match.group(1))
        folder = post_data["/app-api/enduserapp/shared-folder"]
        for item in folder["items"]:
            if item.get("type") == "file" and item.get("extension", "").lower() in VIDEO_EXTENSIONS:
                rows.append({
                    "id": int(item["id"]),
                    "name": item["name"],
                    "size_bytes": int(item["itemSize"]),
                })
        if page >= int(folder["pageCount"]):
            break
        page += 1
    shared_name = post_data["/app-api/enduserapp/shared-item"]["sharedName"]
    table = pd.DataFrame(rows).drop_duplicates("id")
    table["size_GB"] = table["size_bytes"] / 1_000_000_000
    table["start_utc"] = table["name"].map(brightness.parse_utc_start)
    return table.sort_values(["start_utc", "name"]).reset_index(drop=True), shared_name

def download_box_file(shared_name, file_id, destination, block_size=8 * 1024 * 1024):
    query = urllib.parse.urlencode({
        "rm": "box_download_shared_file",
        "shared_name": shared_name,
        "file_id": f"f_{file_id}",
    })
    url = f"https://ucsb.app.box.com/index.php?{query}"
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open("wb") as output:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0
        while True:
            block = response.read(block_size)
            if not block:
                break
            output.write(block)
            downloaded += len(block)
            if total:
                print(f"\rDownloaded {downloaded / total:6.1%}", end="")
    print()
    return destination

## List the remote videos

This downloads only the small Box folder pages—not the videos. Inspect `size_GB` before selecting a batch.

In [ ]:
videos, shared_name = list_box_videos(BOX_FOLDER_URL)
print(f"Found {len(videos)} videos ({videos['size_GB'].sum():.2f} GB total)")
display(videos[["name", "size_GB", "start_utc"]].style.format({"size_GB": "{:.3f}"}))

## Select and process a batch

Adjust `BATCH_START` and `BATCH_SIZE` in the first code cell, then inspect this selection. Already-completed videos are skipped. During processing, only one downloaded video is present locally. Its temporary directory is removed even if scanning raises an error or you interrupt the cell.

In [ ]:
selected = videos.iloc[BATCH_START:BATCH_START + BATCH_SIZE].copy()
selected["result_exists"] = selected["name"].map(
    lambda name: (RESULTS_DIR / f"{Path(name).stem}_{VIDEO_REGION}_brightness.csv").exists()
)
display(selected[["name", "size_GB", "start_utc", "result_exists"]].style.format({"size_GB": "{:.3f}"}))

In [ ]:
local_zone = ZoneInfo(TIMEZONE)

for item in selected.itertuples(index=False):
    result_path = RESULTS_DIR / f"{Path(item.name).stem}_{VIDEO_REGION}_brightness.csv"
    if result_path.exists() and not FORCE_REPROCESS:
        print(f"Skipping completed video: {item.name}")
        continue

    start_utc = brightness.parse_utc_start(item.name)
    if start_utc is None:
        print(f"Skipping filename without a recognizable UTC time: {item.name}")
        continue

    print(f"\nProcessing {item.name} ({item.size_GB:.3f} GB)")
    with tempfile.TemporaryDirectory(prefix="manta_brightness_") as temp_dir:
        local_video = Path(temp_dir) / item.name
        download_box_file(shared_name, item.id, local_video)
        measurements = brightness.scan_video(
            local_video,
            start_utc,
            local_zone,
            FRAME_STEP,
            MAX_WIDTH,
            TOP_FRACTION,
            PERCENTILE,
            BRIGHT_THRESHOLD,
            VIDEO_REGION,
        )
        brightness.write_csv(measurements, result_path)
        print(f"Saved {len(measurements):,} measurements to {result_path.name}")
    print("Deleted the temporary local video.")

## Combine completed results and plot Panama clock time

This cell reads every completed per-video CSV, not only the most recent batch. The upper panel is whole-frame mean brightness. The lower panel is the mean of the brightest 0.1% of pixels and is more sensitive to small luminous spots.

In [ ]:
result_files = sorted(RESULTS_DIR.glob(f"*_{VIDEO_REGION}_brightness.csv"))
if not result_files:
    raise RuntimeError("No completed result files yet. Run the processing cell first.")

results = pd.concat((pd.read_csv(path) for path in result_files), ignore_index=True)
results["time_panama"] = pd.to_datetime(results["time_utc"], format="mixed", utc=True).dt.tz_convert(TIMEZONE)
results = results.sort_values("time_panama").reset_index(drop=True)
print(f"Loaded {len(results):,} measurements from {len(result_files)} videos")

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, constrained_layout=True)
axes[0].plot(results["time_panama"], results["mean_brightness"], lw=0.7, color="0.25")
axes[0].set_ylabel("Mean brightness (0–255)")
axes[1].plot(results["time_panama"], results["top_fraction_mean"], lw=0.7, color="#0072B2")
axes[1].set_ylabel("Brightest 0.1% mean (0–255)")
axes[1].set_xlabel("Clock time in Panama")
axes[0].set_ylim(0, MEAN_Y_AXIS_MAX)
axes[1].set_ylim(0, BRIGHT_Y_AXIS_MAX)
for axis in axes:
    axis.grid(alpha=0.2)
locator = mdates.AutoDateLocator(minticks=4, maxticks=12)
axes[1].xaxis.set_major_locator(locator)
axes[1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=ZoneInfo(TIMEZONE)))
fig.suptitle("Video brightness screening")
plt.show()

## Download the video closest to a Panama clock time

Enter a local Panama date and time below. The closest video by start time is downloaded into `data/raw/selected_videos/` and retained for closer inspection. Existing downloads are not repeated.

In [55]:
TARGET_PANAMA_TIME = "2026-08-24 03:00:00"  # YYYY-MM-DD HH:MM:SS
SELECTED_VIDEO_DIR = PROJECT_ROOT / "data" / "raw" / "selected_videos"
SELECTED_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

target_time = pd.Timestamp(TARGET_PANAMA_TIME, tz=TIMEZONE)
video_times = pd.to_datetime(videos["start_utc"], utc=True).dt.tz_convert(TIMEZONE)
closest_index = (video_times - target_time).abs().idxmin()
closest = videos.loc[closest_index]
closest_time = video_times.loc[closest_index]
time_difference = abs(closest_time - target_time)
destination = SELECTED_VIDEO_DIR / closest["name"]

print(f"Requested Panama time: {target_time}")
print(f"Closest video start:   {closest_time}")
print(f"Time difference:       {time_difference}")
print(f"Selected video:        {closest['name']} ({closest['size_GB']:.3f} GB)")

if destination.exists():
    print(f"Already downloaded: {destination}")
else:
    download_box_file(shared_name, int(closest["id"]), destination)
    print(f"Saved selected video to {destination}")

destination

Requested Panama time: 2026-08-24 03:00:00-05:00
Closest video start:   2026-08-24 03:06:02-05:00
Time difference:       0 days 00:06:02
Selected video:        sbs_20260824_080602.mkv (0.034 GB)
Downloaded 100.0%
Saved selected video to /Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/sbs_20260824_080602.mkv


PosixPath('/Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/sbs_20260824_080602.mkv')

## Split the selected side-by-side video

This streams the selected recording into separate left- and right-camera MP4 files in `data/raw/selected_videos/split/`. The original downloaded MKV is retained.

In [56]:
import cv2

SPLIT_VIDEO_DIR = SELECTED_VIDEO_DIR / "split"
SPLIT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
left_video = SPLIT_VIDEO_DIR / f"{destination.stem}_left.mp4"
right_video = SPLIT_VIDEO_DIR / f"{destination.stem}_right.mp4"

if left_video.exists() and right_video.exists():
    print("Split videos already exist:")
else:
    capture = cv2.VideoCapture(str(destination))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {destination}")

    fps = capture.get(cv2.CAP_PROP_FPS)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    half_width = width // 2
    if width % 2:
        capture.release()
        raise RuntimeError(f"Video width must be even for a side-by-side split; got {width}")

    codec = cv2.VideoWriter_fourcc(*"mp4v")
    left_writer = cv2.VideoWriter(str(left_video), codec, fps, (half_width, height))
    right_writer = cv2.VideoWriter(str(right_video), codec, fps, (half_width, height))
    if not left_writer.isOpened() or not right_writer.isOpened():
        capture.release()
        left_writer.release()
        right_writer.release()
        raise RuntimeError("Could not create the split MP4 files")

    frame_number = 0
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            left_writer.write(frame[:, :half_width])
            right_writer.write(frame[:, half_width:])
            frame_number += 1
            if frame_number % 500 == 0 or frame_number == total_frames:
                print(f"\rSplit {frame_number:,}/{total_frames:,} frames", end="")
    finally:
        capture.release()
        left_writer.release()
        right_writer.release()
    print()

print(f"Left:  {left_video}")
print(f"Right: {right_video}")
left_video, right_video

Split 17,942/17,942 frames
Left:  /Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/split/sbs_20260824_080602_left.mp4
Right: /Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/split/sbs_20260824_080602_right.mp4


(PosixPath('/Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/split/sbs_20260824_080602_left.mp4'),
 PosixPath('/Users/oakley/Documents/GitHub/manta-analysis/data/raw/selected_videos/split/sbs_20260824_080602_right.mp4'))